# 09 · Perceptron visualisation

This notebook is the **educational part** of lab 09. It goes through the lecture on one neuron: the perceptron rule,
the linear and logistic neuron, and the metrics for classification and regression. Every method runs on small synthetic data
and has a plot with **controls** (sliders, menus, check boxes). You do not need to read or change any code.

**How to work with it:** run all cells (*Run → Run All Cells*). Under every part, controls appear above its plot; each change redraws
the plot. The code cells are collapsed; click the bar to the left of a cell (or the three dots) to expand one if you are curious.
Each part ends with a **✏️ Task**; your answers will help you with tuning in the practical part (`perceptron-practice-en.ipynb`).


If no controls appear, the package `ipywidgets` is missing
(see the README).

In [ ]:
import numpy as np                          # arrays and maths
import pandas as pd                         # result tables
import matplotlib.pyplot as plt             # plots
import ipywidgets as widgets                # sliders, menus and check boxes
from IPython.display import display         # nicely formatted tables
from scipy.stats import norm                # normal distribution (theoretical AUC, ideal probabilities)

BLUE, ORANGE, TEAL, GREY = "#0065BD", "#C2410C", "#00897B", "#8C96A0"  # colours as in the lecture
plt.rcParams.update({"figure.figsize": (11, 3.8), "axes.grid": True, "grid.alpha": 0.3,  # consistent plot style
                     "axes.spines.top": False, "axes.spines.right": False})
SEED = 0                                    # seed: everyone gets the same data
STYLE = {"description_width": "190px"}      # room for the control labels
WIDE = widgets.Layout(width="95%", max_width="560px")  # narrows in a small window


def fslider(label, value, lo, hi, step, fmt=".2f", live=False):
    # slider for a decimal number; live=True redraws while dragging
    return widgets.FloatSlider(value=value, min=lo, max=hi, step=step, description=label, readout_format=fmt,
                               continuous_update=live, style=STYLE, layout=WIDE)


def islider(label, value, lo, hi, step=1, live=False):
    # slider for a whole number
    return widgets.IntSlider(value=value, min=lo, max=hi, step=step, description=label,
                             continuous_update=live, style=STYLE, layout=WIDE)


def choice(label, options, value):
    # slider over a few given values
    return widgets.SelectionSlider(options=options, value=value, description=label,
                                   continuous_update=False, style=STYLE, layout=WIDE)


def menu(label, options, value):
    # drop-down menu; options are (text, value) pairs
    return widgets.Dropdown(options=options, value=value, description=label, style=STYLE, layout=WIDE)


def controls(draw, **inputs):
    # shows the controls and redraws the plot after every change
    widgets.interact(draw, **inputs)


def rng(seed=SEED):
    # a fresh generator with a fixed seed -> the same data after every change
    return np.random.default_rng(seed)


def perceptron(X, y, w0=(0.0, 0.0), b0=0.0, eta=1.0, order=None, epochs=100, g=None, bias=True):
    # the perceptron rule from the lecture: y ∈ {−1, +1}, a sample is an error when y·s ≤ 0
    w, b = np.array(w0, dtype=float), float(b0)
    history = [(w.copy(), b)]               # weights after every update
    errors_per_epoch = []
    for epoch in range(epochs):
        if g is not None:
            idx = g.permutation(len(y))     # random order in every epoch
        else:
            idx = order if order is not None else range(len(y))  # fixed order
        errors = 0
        for i in idx:
            s = w @ X[i] + b                # score of the sample
            if y[i] * s <= 0:               # wrong side of the line or exactly on it
                w = w + eta * y[i] * X[i]   # move the line towards the sample
                if bias:
                    b = b + eta * y[i]
                errors += 1
                history.append((w.copy(), b))
        errors_per_epoch.append(errors)
        if errors == 0:                     # a whole epoch without an error: stop
            break
    return w, b, errors_per_epoch, history


def count_errors(X, y, w, b):
    # number of training samples on the wrong side (or on the line)
    return int(np.sum(y * (X @ w + b) <= 0))


def draw_line(ax, w, b, xlim, **style):
    # draws the line w₁x₁ + w₂x₂ + b = 0
    x = np.array(xlim, dtype=float)
    if abs(w[1]) > 1e-12:
        ax.plot(x, -(w[0] * x + b) / w[1], **style)
    elif abs(w[0]) > 1e-12:
        ax.axvline(-b / w[0], **style)      # vertical line when w₂ = 0


def show_classes(ax, X, y, s=30, alpha=0.9):
    # class +1 (or 1) as orange diamonds, class −1 (or 0) as blue circles
    pos = y > 0
    ax.scatter(*X[~pos].T, s=s, color=BLUE, alpha=alpha, label="class −1" if y.min() < 0 else "class 0")
    ax.scatter(*X[pos].T, s=s * 1.2, color=ORANGE, marker="D", alpha=alpha, label="class +1" if y.min() < 0 else "class 1")


def two_clouds(g, n, distance, spread=1.0, gap=None):
    # two Gaussian clouds on the diagonal whose centres are `distance` apart
    c = np.array([1, 1]) / np.sqrt(2) * distance / 2
    X = np.r_[g.normal(c, spread, (n, 2)), g.normal(-c, spread, (n, 2))]
    y = np.r_[np.ones(n), -np.ones(n)].astype(int)
    if gap is not None:                     # remove points from a band around x₁ + x₂ = 0 -> separable data
        keep = y * (X @ np.array([1, 1])) / np.sqrt(2) > gap / 2
        X, y = X[keep], y[keep]
    return X, y


print("Ready. Run the following cells; controls will appear under each part.")

## 1. The perceptron rule step by step

**Goal of the method:** find weights **w** and a bias b such that the line w₁x₁ + w₂x₂ + b = 0 separates two classes.
The score of a sample is s = **w**·**x** + b, the prediction is the sign of s. Samples are visited one by one; when a sample
is misclassified (y·s ≤ 0), the line moves towards it:

**w** ← **w** + η·y·**x**, b ← b + η·y.

The data are the four points from the lecture: x₁ = (3, 2) and x₃ = (1, 3) in class +1, x₂ = (0, −2) and x₄ = (−1, 2) in class −1.
With η = 1, a zero start and the order x₁, x₂, x₃, x₄ the lecture result is **w** = (4, 2), b = −1 after 3 updates.

**Controls:** *step* moves through the evaluations one by one (the ringed point is the sample just evaluated, the dashed line
the previous boundary); the other controls change the learning rate, the start and the order. The table lists the evaluations up to the current step.

In [ ]:
X1 = np.array([[3, 2], [0, -2], [1, 3], [-1, 2]], dtype=float)  # the four points from the lecture
y1 = np.array([1, -1, 1, -1])                                    # their classes


def section1(step, eta, start, order):
    (w0, b0) = start
    w, b = np.array(w0, dtype=float), float(b0)
    log = []                                     # one record per evaluation
    for epoch in range(1, 21):
        errors = 0
        for i in order:
            s = w @ X1[i] + b
            before = (w.copy(), b)
            wrong = y1[i] * s <= 0
            if wrong:
                w, b = w + eta * y1[i] * X1[i], b + eta * y1[i]
                errors += 1
            log.append({"epoch": epoch, "sample": f"x{i + 1}", "y": y1[i], "s": s,
                        "update": "yes" if wrong else "", "w₁": w[0], "w₂": w[1], "b": b,
                        "_i": i, "_before": before, "_after": (w.copy(), b)})
        if errors == 0:
            break
    k = min(step, len(log))                      # after convergence nothing changes
    w_k, b_k = log[k - 1]["_after"] if k else (np.array(w0, dtype=float), float(b0))

    fig, ax = plt.subplots(figsize=(6, 6))
    for rec in log[:k]:                          # earlier boundaries, faint
        draw_line(ax, *rec["_after"], (-3, 5), color=TEAL, alpha=0.15, lw=1)
    if k:
        draw_line(ax, *log[k - 1]["_before"], (-3, 5), color=GREY, ls="--", lw=1.5, label="before this step")
        cur = X1[log[k - 1]["_i"]]
        ax.scatter(*cur, s=600, facecolors="none", edgecolors="black", lw=2)   # the sample just evaluated
    draw_line(ax, w_k, b_k, (-3, 5), color=TEAL, lw=3, label="current boundary")
    show_classes(ax, X1, y1, s=90)
    for i, (xa, xb) in enumerate(X1):
        ax.annotate(f"x{i + 1}", (xa, xb), textcoords="offset points", xytext=(8, 6), fontsize=12)
    ax.set(xlim=(-3, 5), ylim=(-4, 5), xlabel="x₁", ylabel="x₂",
           title=f"step {k}/{len(log)}: w = ({w_k[0]:g}, {w_k[1]:g}), b = {b_k:g}")
    ax.set_aspect("equal")
    ax.legend(loc="lower right", fontsize=9)
    plt.show()
    table = pd.DataFrame(log[:k]).drop(columns=["_i", "_before", "_after"]) if k else pd.DataFrame()
    display(table)
    updates = sum(r["update"] != "" for r in log)
    w_end, b_end = log[-1]["_after"]
    print(f"at the end: w = ({w_end[0]:g}, {w_end[1]:g}), b = {b_end:g} after "
          f"{updates} updates, smallest distance of a point from the line "
          f"{(np.abs(X1 @ w_end + b_end) / np.linalg.norm(w_end)).min():.2f}")
    if (eta, start, order) == (1.0, ((0.0, 0.0), 0.0), (0, 1, 2, 3)):
        assert np.allclose(w_end, [4, 2]) and b_end == -1 and updates == 3   # the lecture result


controls(section1,
         step=islider("step (evaluation)", 4, 0, 16, live=True),
         eta=choice("learning rate η", [0.25, 0.5, 1.0, 2.0], 1.0),
         start=menu("start w, b", [("w = (0, 0), b = 0", ((0.0, 0.0), 0.0)), ("w = (−1, 1), b = 0", ((-1.0, 1.0), 0.0)),
                                                 ("w = (1, −1), b = 1", ((1.0, -1.0), 1.0))], ((0.0, 0.0), 0.0)),
         order=menu("order in an epoch", [("x₁, x₂, x₃, x₄", (0, 1, 2, 3)), ("x₂, x₁, x₃, x₄", (1, 0, 2, 3)),
                                                          ("x₃, x₁, x₂, x₄", (2, 0, 1, 3)), ("x₄, x₃, x₂, x₁", (3, 2, 1, 0))], (0, 1, 2, 3)))

**✏️ Task 1**

* Step through the first two epochs (steps 1–8) and compare with the lecture. Why does x₂ count as an error in step 6 although its score is not positive?
* Choose the order x₂, x₁, x₃, x₄ and move the step to the end. Which line do you get? Is it also a correct solution? Which of the two lines has the larger smallest distance?
* Set η = 0.5 with the zero start. How do the weights change and how does the line change? Why? Then choose the start w = (−1, 1): does η matter now?

## 2. Convergence and the bound on the number of updates

**Goal of the method:** know when the perceptron is guaranteed to stop. If a line separates the classes with a **margin** γ
(no point is closer to it than γ) and all points lie within a radius R, the perceptron makes at most **(R/γ)²** updates
(Novikoff's theorem). The bound does not depend on the number of points or on their order.

The points lie in a disc of radius 3 and are separated by the line x₁ + x₂ = 0 with an empty band of half-width γ.
The line goes through the origin, so the perceptron is trained without a bias. Ten runs, each with new data and a random order.

**Control:** the margin γ.

In [ ]:
def disc_data(g, gamma, n=200, radius=3.0):
    # points in a disc, without the band |d| ≤ γ around the line x₁ + x₂ = 0
    P = g.uniform(-radius, radius, (n * 4, 2))
    P = P[np.linalg.norm(P, axis=1) <= radius]
    d = P @ np.array([1, 1]) / np.sqrt(2)             # signed distance from the line
    keep = np.abs(d) > gamma
    P, d = P[keep][:n], d[keep][:n]
    return P, np.where(d > 0, 1, -1)


def section2(gamma):
    updates, bounds = [], []
    for run in range(10):
        X, y = disc_data(rng(100 + run), gamma)
        w, b, errs, hist = perceptron(X, y, g=rng(run), bias=False, epochs=5000)
        R = np.linalg.norm(X, axis=1).max()                         # radius of the data
        margin = np.min(np.abs(X @ np.array([1, 1])) / np.sqrt(2))  # actual margin of the true line
        updates.append(len(hist) - 1)
        bounds.append((R / margin) ** 2)
    fig, axs = plt.subplots(1, 2, gridspec_kw={"width_ratios": [1, 1.4]})
    t = np.array([-3, 3])
    axs[0].fill_between(t, -t - gamma * np.sqrt(2), -t + gamma * np.sqrt(2), color=GREY, alpha=0.25, label="band ±γ")
    show_classes(axs[0], X, y, s=12)
    draw_line(axs[0], w, 0.0, (-3, 3), color=TEAL, lw=2, label="perceptron (last run)")
    axs[0].set(xlim=(-3.2, 3.2), ylim=(-3.2, 3.2), xlabel="x₁", ylabel="x₂")
    axs[0].set_aspect("equal")
    axs[0].legend(loc="lower left", fontsize=8)
    axs[1].bar(range(1, 11), updates, color=BLUE, label="updates until convergence")
    axs[1].axhline(np.mean(bounds), color=ORANGE, ls="--", label=f"bound (R/γ)² ≈ {np.mean(bounds):.0f}")
    axs[1].set(xlabel="run", ylabel="number of updates", yscale="log", xticks=range(1, 11))
    axs[1].legend()
    plt.tight_layout()
    plt.show()
    print(f"γ = {gamma}: mean number of updates {np.mean(updates):.0f}, "
          f"maximum {max(updates)}, bound {np.mean(bounds):.0f}")


controls(section2, gamma=choice("margin γ", [0.05, 0.1, 0.2, 0.5, 1.0], 0.2))

**✏️ Task 2**

* How far is the actual number of updates from the bound? Is the bound ever exceeded?
* Halve γ (0.2 → 0.1 → 0.05). How many times does the bound grow? And the mean number of updates?
* Is the perceptron line the same as the true line x₁ + x₂ = 0? Why not?

## 3. Linearly non-separable data and the pocket algorithm

**Goal of the method:** obtain a usable line even when no line separates the classes. The perceptron rule never stops
on such data: every epoch contains errors and every error moves the line. The **pocket algorithm** keeps aside
(“in the pocket”) the weights with the fewest training errors seen so far and returns them at the end.

Two clouds of points overlap; the overlap is set by the distance of their centres. The left plot compares the last weights
with the pocket weights, the right plot shows the training errors after every update.

**Controls:** distance of the cloud centres (small = strong overlap), number of epochs.

In [ ]:
def section3(distance, epochs):
    X, y = two_clouds(rng(13), 60, distance)
    g = rng(14)
    w, b = np.zeros(2), 0.0
    best_w, best_b, best_errors = w.copy(), b, count_errors(X, y, w, b)
    current, pocket = [], []
    for epoch in range(epochs):
        for i in g.permutation(len(y)):
            if y[i] * (w @ X[i] + b) <= 0:             # an ordinary perceptron update
                w, b = w + y[i] * X[i], b + y[i]
                e = count_errors(X, y, w, b)           # errors of the new weights on all training data
                if e < best_errors:                    # better than the pocket -> into the pocket
                    best_w, best_b, best_errors = w.copy(), b, e
                current.append(e)
                pocket.append(best_errors)
    fig, axs = plt.subplots(1, 2, gridspec_kw={"width_ratios": [1, 1.5]})
    show_classes(axs[0], X, y, s=16)
    draw_line(axs[0], w, b, (-5, 5), color=GREY, lw=2, ls="--", label="last weights")
    draw_line(axs[0], best_w, best_b, (-5, 5), color=TEAL, lw=2.5, label="pocket weights")
    axs[0].set(xlim=(-5, 5), ylim=(-5, 5), xlabel="x₁", ylabel="x₂")
    axs[0].set_aspect("equal")
    axs[0].legend(loc="lower right", fontsize=8)
    axs[1].plot(current, color=GREY, lw=1, label="current weights")
    axs[1].step(range(len(pocket)), pocket, where="post", color=BLUE, lw=2.5, label="pocket weights")
    axs[1].set(xlabel="number of updates", ylabel="training errors")
    axs[1].legend()
    plt.tight_layout()
    plt.show()
    print(f"updates: {len(current)}, errors of the last weights: {count_errors(X, y, w, b)}, "
          f"errors in the pocket: {best_errors} of {len(y)}")


controls(section3,
         distance=fslider("distance of the centres", 2.5, 1.0, 6.0, 0.5, fmt=".1f"),
         epochs=islider("epochs", 20, 1, 100))

**✏️ Task 3**

* Does the number of errors of the current weights settle as the epochs go on? Are the last weights the best ones?
* Increase the epochs to 100. Does the pocket improve? Do the last weights improve?
* Set the distance to 6. What happens to the number of updates? Why does the plain perceptron rule have no way to stop on overlapping data?

## 4. Input scaling

**Goal of the method:** give the features comparable scales so that every update moves all weights by comparable amounts.
A perceptron update adds η·y·**x**: a feature with values in thousands changes its weight a thousand times more than a feature
in units, while the bias changes only by η. **Standardisation** subtracts the mean and divides by the standard deviation of each feature.

The same separable data are learned twice, with the second feature in original units (spread and offset) and after standardisation.

**Controls:** how many times larger the spread of x₂ is, and the offset added to x₂.

In [ ]:
def section4(scale_x2, offset_x2):
    Z, y = two_clouds(rng(15), 50, 3.4, gap=0.3)                    # separable data as in the lecture
    X_raw = np.c_[Z[:, 0], offset_x2 + scale_x2 * Z[:, 1]]          # x₂ in “large” units
    X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)        # standardisation
    curves = {}
    for name, Xq in (("original units", X_raw), ("standardised", X_std)):
        g, w, b, errs = rng(16), np.zeros(2), 0.0, []
        for epoch in range(40):
            w, b, _, _ = perceptron(Xq, y, w0=w, b0=b, g=g, epochs=1)  # one epoch, continuing from the previous weights
            errs.append(count_errors(Xq, y, w, b))
        curves[name] = errs
    fig, axs = plt.subplots(1, 2)
    show_classes(axs[0], X_raw, y, s=14)
    axs[0].set(xlabel="x₁", ylabel="x₂ in original units")
    for (name, errs), color, marker in zip(curves.items(), (ORANGE, BLUE), ("^", "o")):
        axs[1].plot(range(1, 41), errs, color=color, marker=marker, ms=4, label=name)
    axs[1].set(xlabel="epoch", ylabel="training errors after the epoch")
    axs[1].legend()
    plt.tight_layout()
    plt.show()
    for name, errs in curves.items():
        print(f"{name}: errors in the last 10 epochs {errs[-10:]}")


controls(section4,
         scale_x2=choice("spread of x₂ (times)", [1, 10, 200, 1000], 200),
         offset_x2=menu("offset of x₂", [("0", 0), ("1000", 1000)], 1000))

**✏️ Task 4**

* Which version reaches zero training errors, and after how many epochs? What does the unscaled version do instead, and why?
* Set the offset to 0 and keep the spread 200. Does the offset or the spread hurt more? Hint: how large is the bias update compared with x₂?
* With spread 1 and offset 0 both curves coincide almost exactly. Why?

## 5. Linear neuron: gradient descent and the learning rate

**Goal of the method:** fit a line ŷ = w·x + b to continuous targets by minimising the **mean squared error**
E = (1/n)·Σ(y − ŷ)². Gradient descent repeats the step (w, b) ← (w, b) − η·∇E. A step that is too small converges slowly,
a step that is too large jumps over the minimum and diverges. For the squared error the limit can be computed:
the descent converges for η < 1/λ_max, where λ_max is the largest eigenvalue of the matrix of moments [[mean x², mean x], [mean x, 1]].

The data are the 40 regression points from the lecture (the interactive slide on the learning rate). The left plot shows the contours
of E over (w, b) and the path of the descent from (−0.5, 5), the right plot shows E after every step.

**Controls:** learning rate η, number of steps, unit of x (the same data with x in ten-times-larger numbers).

In [ ]:
g = rng(17)
x5 = g.uniform(0, 4, 40)                        # the regression data from the lecture
y5 = 1.5 * x5 + 1 + g.normal(0, 0.6, 40)


def section5(eta, steps, x_unit):
    x, y = x5 * x_unit, y5                      # the same data in a different unit
    M = np.array([[np.mean(x ** 2), np.mean(x)], [np.mean(x), 1.0]])  # matrix of moments
    eta_max = 1 / np.linalg.eigvalsh(M).max()   # stability limit of the learning rate
    w_opt, b_opt = np.polyfit(x, y, 1)          # exact minimum for comparison
    if x_unit == 1:
        assert round(eta_max, 2) == 0.18        # the value from the lecture
    mse = lambda w, b: np.mean((w * x + b - y) ** 2)
    path, errors = [(-0.5 / x_unit, 5.0)], []
    with np.errstate(over="ignore", invalid="ignore"):
        for step in range(steps):
            w, b = path[-1]
            errors.append(mse(w, b))
            r = w * x + b - y                   # residuals with the opposite sign
            grad = np.array([2 * np.mean(r * x), 2 * np.mean(r)])  # gradient of E
            path.append(tuple(np.array([w, b]) - eta * grad))
            if not np.isfinite(path[-1]).all() or abs(path[-1][1]) > 1e6:
                break                           # diverged: stop drawing
    path = np.array(path)
    W, Bb = np.meshgrid(np.linspace(w_opt - 2.5 / x_unit, w_opt + 2.5 / x_unit, 200), np.linspace(b_opt - 5, b_opt + 5, 200))
    E = np.mean((W[..., None] * x + Bb[..., None] - y) ** 2, axis=-1)
    fig, axs = plt.subplots(1, 2)
    axs[0].contour(W, Bb, E, levels=np.quantile(E, [0.002, 0.01, 0.03, 0.07, 0.15, 0.3, 0.5, 0.75]), colors=GREY, linewidths=1)
    axs[0].plot(path[:, 0], path[:, 1], color=ORANGE if eta >= eta_max else BLUE, marker="o", ms=3, lw=1.5)
    axs[0].scatter([w_opt], [b_opt], marker="*", s=200, color="black", zorder=5)
    axs[0].set(xlim=(W.min(), W.max()), ylim=(Bb.min(), Bb.max()), xlabel="w", ylabel="b", title=f"η = {eta:g}, η_max ≈ {eta_max:.3g}")
    axs[1].semilogy(range(len(errors)), errors, color=BLUE, marker="o", ms=3)
    axs[1].axhline(mse(w_opt, b_opt), color=GREY, ls="--", label="minimum of E")
    axs[1].set(xlabel="step", ylabel="E (log scale)")
    axs[1].legend()
    plt.tight_layout()
    plt.show()
    print(f"limit η_max = 1/λ_max ≈ {eta_max:.4f}; E after the last step {errors[-1]:.4g}, "
          f"minimum {mse(w_opt, b_opt):.4g}")


controls(section5,
         eta=fslider("learning rate η", 0.12, 0.005, 0.2, 0.005, fmt=".3f"),
         steps=islider("number of steps", 40, 10, 200, 10),
         x_unit=menu("unit of x", [("original", 1), ("x ten times larger", 10)], 1))

**✏️ Task 5**

* Find the smallest η at which the descent diverges. How close is it to the printed limit?
* Between roughly 0.09 and 0.18 the path zigzags but still converges. Along which direction does it zigzag, and why that one?
* Switch the unit of x to “ten times larger” (the same data in different units). How did η_max change? What does this mean for the choice of η without scaling?

## 6. Logistic neuron and log loss

**Goal of the method:** obtain a **probability** of class 1 instead of a hard decision. The logistic neuron passes the score
through the sigmoid, p = σ(s) = 1 / (1 + e^(−s)), and learns by minimising the **log loss** L = −ln p_y, where p_y is the
probability the model gives to the correct class. The gradient has the same form as the delta rule: (p − y)·**x**.

The model is trained by batch gradient descent on two overlapping clouds (classes 0 and 1). The left plot shows the probability
contours 0.1, 0.5 and 0.9, the middle the mean log loss after every epoch, the right the loss of each sample against its score.

**Controls:** learning rate, number of epochs (gradient steps over all data), distance of the cloud centres (from about 6 the classes are separable).

In [ ]:
def sigmoid(s):
    return 1 / (1 + np.exp(-s))


def section6(eta, epochs, distance):
    X, y_pm = two_clouds(rng(22), 60, distance)
    y = (y_pm > 0).astype(int)                       # classes 0 and 1
    w, b, losses = np.zeros(2), 0.0, []
    with np.errstate(over="ignore", divide="ignore"):
        for epoch in range(epochs):
            p = sigmoid(X @ w + b)                   # probability of class 1
            p_correct = np.where(y == 1, p, 1 - p)   # probability of the correct class
            losses.append(np.mean(-np.log(np.clip(p_correct, 1e-12, 1))))
            w = w - eta * np.mean((p - y)[:, None] * X, axis=0)  # gradient step: (p − y)·x
            b = b - eta * np.mean(p - y)
        xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
        P = sigmoid(np.c_[xx.ravel(), yy.ravel()] @ w + b).reshape(xx.shape)
        s = X @ w + b
        loss_each = -np.log(np.clip(np.where(y == 1, sigmoid(s), 1 - sigmoid(s)), 1e-12, 1))
        fig, axs = plt.subplots(1, 3, figsize=(13, 4))
        cs = axs[0].contour(xx, yy, P, levels=[0.1, 0.5, 0.9], colors=[BLUE, "black", ORANGE])
        axs[0].clabel(cs, fontsize=9)
        show_classes(axs[0], X, y, s=14)
        axs[0].set(xlabel="x₁", ylabel="x₂", xlim=(-5, 5), ylim=(-5, 5))
        axs[0].set_aspect("equal")
        axs[1].plot(losses, color=BLUE)
        axs[1].set(xlabel="epoch", ylabel="mean log loss", ylim=(0, None))
        grid_s = np.linspace(min(-6, s.min()), max(6, s.max()), 300)
        axs[2].plot(grid_s, -np.log(sigmoid(grid_s)), color=ORANGE, label="y = 1: −ln σ(s)")
        axs[2].plot(grid_s, -np.log(1 - sigmoid(grid_s)), color=BLUE, label="y = 0: −ln(1 − σ(s))")
        axs[2].scatter(s, loss_each, c=np.where(y == 1, ORANGE, BLUE), s=10, zorder=3)
        axs[2].set(xlabel="score s", ylabel="loss of the sample", ylim=(0, 6))
        axs[2].legend(fontsize=8)
        plt.tight_layout()
        plt.show()
    print(f"w = ({w[0]:.2f}, {w[1]:.2f}), b = {b:.2f}, ‖w‖ = {np.linalg.norm(w):.2f}, final mean loss {losses[-1]:.3f}")
    print("loss for p_y = 0.9, 0.5, 0.01:", [round(float(-np.log(q)), 2) for q in (0.9, 0.5, 0.01)])


controls(section6,
         eta=choice("learning rate", [0.01, 0.1, 0.5, 2.0, 5.0, 20.0], 0.5),
         epochs=choice("epochs", [10, 50, 200, 1000, 2000, 5000], 200),
         distance=fslider("distance of the centres", 2.5, 1.0, 8.0, 0.5, fmt=".1f"))

**✏️ Task 6**

* Which samples contribute most to the loss? What is the loss of a sample to which the model gives only 1 % for its correct class?
* Set the learning rate to 20. Does the loss still decrease? Compare with part 5: why is there no sharp divergence here?
* Set the distance to 8 (separable classes) and increase the epochs to 2000 and 5000. What happens to ‖w‖ and to the width of the band between the 0.1 and 0.9 contours? Why does this call for weight regularisation?

## 7. Confusion matrix and the decision threshold

**Goal of the method:** describe *which* errors a classifier makes. The **confusion matrix** counts true negatives (TN),
false positives (FP), false negatives (FN) and true positives (TP). From it:

| Metric | Formula | Question it answers |
|---|---|---|
| accuracy | (TP + TN) / n | how many predictions are correct |
| precision | TP / (TP + FP) | how many alarms are real |
| recall (sensitivity) | TP / (TP + FN) | how many positives are found |
| F1 | 2·P·R / (P + R) | harmonic mean of precision and recall |
| specificity | TN / (TN + FP) | how many negatives are left alone |
| balanced accuracy | (recall + specificity) / 2 | accuracy that ignores class shares |

A model gives a score; the **threshold** turns it into a decision. The scores of negatives follow N(0, 1), those of positives
N(2, 1), as on the interactive threshold slide in the lecture. The first line checks the metric function on the lecture example
(TN 940, FP 10, FN 20, TP 30).

**Controls:** share of positives, threshold (the plot follows the slider while you drag it).

In [ ]:
ACC, PREC, REC, SPEC, BAL = "accuracy", "precision", "recall", "specificity", "balanced accuracy"


def metrics(tn, fp, fn, tp):
    # all metrics from the four counts of the confusion matrix
    precision = tp / (tp + fp) if tp + fp else float("nan")   # undefined when there are no alarms
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {ACC: (tp + tn) / (tn + fp + fn + tp), PREC: precision, REC: recall,
            "F1": f1, SPEC: specificity, BAL: (recall + specificity) / 2}


lecture = metrics(940, 10, 20, 30)
assert round(lecture[ACC], 3) == 0.97 and round(lecture["F1"], 3) == 0.667 and round(lecture[BAL], 3) == 0.795
print("lecture example:", {k: round(v, 3) for k, v in lecture.items()})


def section7(prevalence, threshold):
    g = rng(26)
    n_pos = int(1000 * prevalence)
    scores = np.r_[g.normal(2, 1, n_pos), g.normal(0, 1, 1000 - n_pos)]
    y = np.r_[np.ones(n_pos), np.zeros(1000 - n_pos)].astype(int)

    def confusion(t):
        pred = scores >= t
        return (int(np.sum(~pred & (y == 0))), int(np.sum(pred & (y == 0))),
                int(np.sum(~pred & (y == 1))), int(np.sum(pred & (y == 1))))

    tn, fp, fn, tp = confusion(threshold)
    table = pd.DataFrame([metrics(tn, fp, fn, tp), metrics(1000 - n_pos, 0, n_pos, 0)],
                         index=[f"threshold {threshold:.2f}", "always 0"]).round(3)
    fig, axs = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw={"width_ratios": [1.4, 0.8, 1.4]})
    bins = np.linspace(-4, 6, 41)
    axs[0].hist(scores[y == 0], bins=bins, color=BLUE, alpha=0.6, label="negatives (y = 0)")
    axs[0].hist(scores[y == 1], bins=bins, color=ORANGE, alpha=0.6, label="positives (y = 1)")
    axs[0].axvline(threshold, color="black", lw=2)
    axs[0].set(xlabel="score", ylabel="count")
    axs[0].legend(fontsize=8)
    axs[1].imshow([[tn, fp], [fn, tp]], cmap="Blues")
    for (r, c), v in np.ndenumerate(np.array([[tn, fp], [fn, tp]])):
        axs[1].text(c, r, f"{['TN', 'FP', 'FN', 'TP'][2 * r + c]}\n{v}", ha="center", va="center", fontsize=12)
    axs[1].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["ŷ = 0", "ŷ = 1"], yticklabels=["y = 0", "y = 1"])
    axs[1].grid(False)
    ths = np.linspace(-2, 4, 121)
    curves = pd.DataFrame([metrics(*confusion(t)) for t in ths], index=ths)
    for name, color in ((ACC, GREY), (PREC, BLUE), (REC, ORANGE), ("F1", TEAL)):
        axs[2].plot(ths, curves[name], color=color, label=name)
    axs[2].axvline(threshold, color="black", lw=1)
    axs[2].set(xlabel="threshold", ylim=(0, 1.02))
    axs[2].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    display(table)
    print(f"highest F1 at threshold {curves['F1'].idxmax():.2f}, highest accuracy at threshold {curves[ACC].idxmax():.2f}")


controls(section7,
         prevalence=fslider("share of positives", 0.2, 0.01, 0.5, 0.01),
         threshold=fslider("threshold", 1.0, -2.0, 4.0, 0.05, live=True))

**✏️ Task 7**

* Set the share of positives to 0.05. Compare the accuracy of the model and of “always 0”. Which metrics reveal that “always 0” is useless?
* Move the threshold from 2 to 0. How do precision and recall change? Why must they move in opposite directions?
* Do the thresholds with the highest F1 and the highest accuracy coincide? Which would you use if a missed positive is ten times worse than a false alarm?

## 8. ROC, precision–recall and the cost of errors

**Goal of the methods:** evaluate a model over **all** thresholds and then choose one threshold by the consequences of errors.

- The **ROC curve** plots recall (TPR) against the false positive rate FPR = 1 − specificity. **AUC** is the area under it:
  the probability that a random positive gets a higher score than a random negative. For two normal distributions with distance μ
  it is Φ(μ/√2).
- The **precision–recall curve** plots precision against recall; its baseline is the share of positives.
- **Cost of errors:** with a cost c_FN for a missed positive and c_FP for a false alarm, the cheapest decision for a **calibrated**
  probability p is “positive when p ≥ c_FP / (c_FP + c_FN)”.

The probabilities here are exact (computed from the known distributions), so they are calibrated by construction.

**Controls:** distance μ of the mean scores of the classes, share of positives, cost of a missed positive c_FN, cost of a false alarm c_FP.

In [ ]:
def section8(distance, prevalence, c_fn, c_fp):
    g = rng(27)
    n = 5000
    n_pos = int(n * prevalence)
    scores = np.r_[g.normal(distance, 1, n_pos), g.normal(0, 1, n - n_pos)]
    y = np.r_[np.ones(n_pos), np.zeros(n - n_pos)].astype(int)
    like_pos = prevalence * norm.pdf(scores, distance, 1)           # Bayes' rule with the known distributions
    p = like_pos / (like_pos + (1 - prevalence) * norm.pdf(scores, 0, 1))  # exact probability of class 1
    order = np.argsort(-scores)                     # thresholds from the highest score down
    tp = np.cumsum(y[order] == 1)
    fp = np.cumsum(y[order] == 0)
    tpr, fpr = np.r_[0, tp / n_pos], np.r_[0, fp / (n - n_pos)]
    precision = tp / (tp + fp)
    auc = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)  # area under the ROC curve
    cuts = np.round(np.arange(0.01, 1.0, 0.01), 2)  # thresholds on the probability
    cost = np.array([c_fn * np.sum((p < t) & (y == 1)) + c_fp * np.sum((p >= t) & (y == 0)) for t in cuts])
    best = cuts[np.argmin(cost)]
    theory = c_fp / (c_fp + c_fn)
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    axs[0].plot(fpr, tpr, color=BLUE, lw=2)
    axs[0].plot([0, 1], [0, 1], color=GREY, ls="--")
    axs[0].set(xlabel="FPR = 1 − specificity", ylabel="recall (TPR)", title=f"ROC, AUC = {auc:.3f}")
    axs[1].plot(tp / n_pos, precision, color=ORANGE, lw=2)
    axs[1].axhline(prevalence, color=GREY, ls="--", label="share of positives")
    axs[1].set(xlabel="recall", ylabel="precision", ylim=(0, 1.02), title="precision–recall")
    axs[1].legend(fontsize=8)
    axs[2].plot(cuts, cost, color=TEAL, lw=2)
    axs[2].axvline(theory, color=ORANGE, ls="--", label=f"c_FP/(c_FP + c_FN) = {theory:.3f}")
    axs[2].scatter([best], [cost.min()], color="black", zorder=3, label=f"minimum at {best:.2f}")
    axs[2].set(xlabel="threshold on the probability p", ylabel="total cost")
    axs[2].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f"AUC {auc:.3f}, theory Φ(μ/√2) = {norm.cdf(distance / np.sqrt(2)):.3f}; "
          f"cheapest threshold {best:.2f} (theory {theory:.3f}), "
          f"cost {cost.min()} vs. threshold 0.5: {cost[cuts == 0.5][0]}, “always 0”: {c_fn * n_pos}")


controls(section8,
         distance=fslider("distance μ", 2.0, 0.0, 4.0, 0.25),
         prevalence=fslider("share of positives", 0.2, 0.01, 0.5, 0.01),
         c_fn=islider("cost of a missed positive c_FN", 10, 1, 20),
         c_fp=islider("cost of a false alarm c_FP", 1, 1, 20))

**✏️ Task 8**

* Change the share of positives from 0.2 to 0.02. Does the ROC curve or AUC change? Does the precision–recall curve? Why the difference?
* How close is the cheapest threshold to c_FP / (c_FP + c_FN)? Set c_FN = 1: where is the optimum now?
* Why does the formula hold only for calibrated probabilities? What would happen with a raw score or with a model trained on rebalanced classes?

## 9. Regression metrics and an outlier

**Goal of the methods:** summarise the residuals e = y − ŷ of a regression model in one number.

| Metric | Formula | Property |
|---|---|---|
| MAE | (1/n)·Σ\|e\| | in units of the target, robust |
| MSE | (1/n)·Σe² | squares: large errors dominate |
| RMSE | √MSE | in units of the target, RMSE ≥ MAE |
| R² | 1 − Σe² / Σ(y − ȳ)² | share of explained variance; 0 = as good as the mean, can be negative |
| MAPE | (100 %/n)·Σ\|e / y\| | relative; unstable for targets near zero |

The data are the ten samples from the lecture (targets 10 to 28, residuals ±1 and ±2). The first row of the table is the
original model, the second your setting.

**Controls:** residual of the last sample (originally −1; the lecture outlier is 12), a shift subtracted from all targets and predictions.

In [ ]:
y9 = np.arange(10, 29, 2, dtype=float)                      # targets 10, 12, …, 28
e9 = np.array([1, -1, 2, -2, 1, -1, 2, -2, 1, -1], dtype=float)  # residuals from the lecture


def regression_metrics(y, y_hat):
    e = y - y_hat
    return {"MAE": np.mean(np.abs(e)), "MSE": np.mean(e ** 2), "RMSE": np.sqrt(np.mean(e ** 2)),
            "R²": 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2), "MAPE %": 100 * np.mean(np.abs(e / y))}


def section9(last_residual, shift):
    e_new = e9.copy()
    e_new[-1] = last_residual
    y_new = y9 - shift
    y_hat_new = y_new - e_new                                  # predictions with the changed residual
    rows = {"original": regression_metrics(y9, y9 - e9), "your setting": regression_metrics(y_new, y_hat_new)}
    table = pd.DataFrame(rows).T.round(3)
    assert tuple(table.iloc[0][["MAE", "MSE", "RMSE", "R²"]]) == (1.4, 2.2, 1.483, 0.933)  # values from the lecture
    fig, axs = plt.subplots(1, 2)
    lim = (min(y_new.min(), y_hat_new.min()) - 2, max(y_new.max(), y_hat_new.max()) + 2)
    axs[0].plot(lim, lim, color=GREY, ls="--")
    axs[0].scatter(y_new, y_hat_new, color=TEAL, s=50)
    axs[0].set(xlim=lim, ylim=lim, xlabel="actual value y", ylabel="prediction ŷ")
    axs[0].set_aspect("equal")
    axs[1].bar(range(1, 11), e_new, color=[ORANGE if abs(v) > 3 else BLUE for v in e_new])
    axs[1].axhline(0, color="black", lw=1)
    axs[1].set(xlabel="sample", ylabel="residual e = y − ŷ", xticks=range(1, 11))
    plt.tight_layout()
    plt.show()
    display(table)


controls(section9,
         last_residual=islider("residual of the last sample", 12, -5, 25, live=True),
         shift=fslider("shift of the targets", 0.0, 0.0, 9.5, 0.5, fmt=".1f", live=True))

**✏️ Task 9**

* With the outlier 12, which metric grows most in relative terms, and why? Which of them would you report to someone who wants “a typical error in units”?
* Find the residual at which R² becomes negative. What does a negative R² mean?
* Set the residual to −1 and the shift to 9.5 (targets close to zero). Which metrics stay the same and which explodes? Why?

## 10. XOR and a new feature

**Goal of the method:** recognise a task that one neuron cannot solve, and one way around it. In XOR, the points (0, 1) and (1, 0)
are in class +1, the points (0, 0) and (1, 1) in class −1. No line separates them, so the perceptron never finishes an epoch without
an update. Adding the **product x₁·x₂** as a third feature makes the classes separable in three dimensions; in the original plane
the boundary is then a curve. The other way – several neurons in layers – is the topic of the lecture on multilayer networks.

**Controls:** add the feature x₁·x₂, number of epochs.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([-1, 1, 1, -1])


def section10(add_product, epochs):
    def features(X):
        # original features, optionally with their product
        return np.c_[X, X[:, 0] * X[:, 1]] if add_product else X

    F = features(X_xor)
    w, b, errs, hist = perceptron(F, y_xor, w0=np.zeros(F.shape[1]), epochs=epochs)
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300), np.linspace(-0.5, 1.5, 300))
    S = (features(np.c_[xx.ravel(), yy.ravel()]) @ w + b).reshape(xx.shape)
    fig, axs = plt.subplots(1, 2)
    axs[0].contourf(xx, yy, S, levels=[-1e9, 0, 1e9], colors=[BLUE, ORANGE], alpha=0.15)
    if S.min() < 0 < S.max():
        axs[0].contour(xx, yy, S, levels=[0], colors="black")
    show_classes(axs[0], X_xor, y_xor, s=120)
    axs[0].set(xlabel="x₁", ylabel="x₂", xticks=[0, 1], yticks=[0, 1],
               title=f"training errors: {count_errors(F, y_xor, w, b)}")
    axs[0].set_aspect("equal")
    axs[1].plot(range(1, len(errs) + 1), errs, color=ORANGE, marker="o")
    axs[1].set(xlabel="epoch", ylabel="updates in the epoch", ylim=(0, 4.5),
               xticks=range(1, len(errs) + 1, max(1, len(errs) // 10)))
    plt.tight_layout()
    plt.show()
    print("w =", np.round(w, 2), " b =", b, " updates per epoch:", errs)


controls(section10,
         add_product=widgets.Checkbox(value=False, description="add the feature x₁·x₂", style=STYLE),
         epochs=islider("epochs", 20, 1, 50))

**✏️ Task 10**

* Why does the number of updates without the product never drop to zero? Would a different order or learning rate help?
* Tick the feature x₁·x₂. After how many epochs does the perceptron stop? What is the sign of the weight of x₁·x₂, and why?
* What does the boundary look like in the original plane? Why can a model with the product feature still be called “linear”?